# 🤖 Chatbot Académico EPIIS-UNSAAC
## IF651 — Inteligencia Artificial | Proyecto Semestral 2026-1 | Segunda Entrega

| Campo | Detalle |
|---|---|
| **Asignatura** | IF651 Inteligencia Artificial |
| **Docentes** | Maritza Katherine Irpanocca Cusimayta / Jisbaj Gamarra Salas |
| **Integrantes** | Castro Pari Rayneld Fidel · Merma Ccarhuarupay Adel Alejandro · Florez Vega Yamir Wagner · Mamani Flores Natan |
| **Atributo** | AG08 Análisis de problemas |
| **Paradigma** | POO — Recuperación basada en corpus + scoring ponderado de keywords |

---
### ▶ Instrucciones de ejecución
1. **Entorno de ejecución → Ejecutar todas las celdas** (Ctrl + F9)
2. Celdas 1-3 crean la carpeta `Repositorio_Conocimiento/` con los 7 JSON
3. Celdas 4-9 definen las 6 clases POO del sistema
4. Celda 10 inicializa y verifica el sistema completo
5. Celda 11 ejecuta los **42 casos de prueba automáticos**
6. Celda 12 abre el **chat interactivo** — escribe `salir` para terminar

---
### Arquitectura del sistema (6 clases POO)
```
[Usuario]
    ↓ texto libre
[ChatbotEPIIS]  ← Fachada (Facade pattern) — única clase expuesta al notebook
    ├─ Preprocessor        lowercase · sin tildes · sin stopwords · tokenización
    ├─ KnowledgeBaseLoader carga 7 JSON · índices O(1) por intent_id
    ├─ IntentMatcher        fase 1: trigger phrases · fase 2: scoring ponderado
    ├─ ResponseBuilder      nivel 1: corpus · nivel 2: enriquecimiento normativo
    └─ ConversationHistory  registra turnos · genera reporte de evaluación
```


## Imports y estructura de carpetas

In [12]:
# ═══════════════════════════════════════════════════
# CELDA 1 — Imports y estructura de carpetas
# ═══════════════════════════════════════════════════
import os, json, re, unicodedata
from datetime import datetime
from pathlib import Path

## Preprocessor con stemming español

In [13]:

# ═══════════════════════════════════════════════════
# CELDA 4 — Preprocessor con stemming español
# ═══════════════════════════════════════════════════
import re, unicodedata


class Preprocessor:
    """
    Normalización avanzada para español:
      - lowercase
      - elimina tildes y signos
      - tokeniza
      - elimina stopwords
      - aplica stemming ligero (raíces)
    """

    STOPWORDS = {
        "la","el","en","de","que","es","un","una","los","las","para","por","con",
        "se","del","al","si","me","mi","tu","su","hay","son","esta","puedo","debo",
        "tengo","como","cuando","donde","cual","cuales","cuantos","cuantas","mas",
        "pero","sin","sobre","entre","hasta","desde","hacia","mis","sus","nos",
        "les","ya","no","ni","yo","a","e","o","u","le","lo","te","ti","ser","era",
        "fue","han","has","he","hemos","estoy","estan","eres","poder","hacer",
        "ver","dar","muy","bien","aqui","ahi","este","esto","esa","ese","eso",
        "porque","esas","esos","estas","estos","unos","unas","mucho","muchos",
        "poco","todo","todos","todas","toda","algo","algunos","alguna","ese",
        "esa","las","los","unas","unos","con","sin","para","quiero","necesito"
    }

    SUFIJOS = [
        "ciones","cion","mente","ando","iendo","ando","iendo","ables","ible",
        "ables","ibles","ados","idos","amos","emos","imos","aron","ieron","ado",
        "ido","aba","ia","ar","er","ir","as","es","os","an","en","al","or"
    ]

    def normalize(self, text):
        text = text.lower().strip()
        text = unicodedata.normalize("NFD", text)
        text = "".join(c for c in text if unicodedata.category(c) != "Mn")
        text = re.sub(r"[^\w\s]", " ", text)
        return re.sub(r"\s+", " ", text).strip()

    def stem(self, word):
        """Stemming ligero: remueve sufijos comunes del español."""
        for suf in self.SUFIJOS:
            if word.endswith(suf) and len(word) - len(suf) >= 4:
                return word[:-len(suf)]
        return word

    def tokenize(self, text):
        normalized = self.normalize(text)
        toks = [t for t in normalized.split()
                if t not in self.STOPWORDS and len(t) > 2]
        return toks

    def process(self, text):
        """Retorna (normalizado, tokens, tokens_con_stems)."""
        normalized = self.normalize(text)
        tokens = self.tokenize(text)
        # Expansión con stems
        expanded = set(tokens)
        for t in tokens:
            stem = self.stem(t)
            if stem != t and len(stem) > 2:
                expanded.add(stem)
        return normalized, set(tokens), expanded


## KnowledgeBaseLoader

In [14]:

# ═══════════════════════════════════════════════════
# CELDA 5 — KnowledgeBaseLoader (sin cambios)
# ═══════════════════════════════════════════════════
import json
from pathlib import Path


class KnowledgeBaseLoader:
    def __init__(self, base_dir="."):
        self._base = Path(base_dir)
        self._corpus = []
        self._intents = []
        self._keywords = []
        self._faq_index = {}
        self._tutorias = {}
        self._practicas = {}
        self._servicios = {}
        self._corpus_by_id = {}
        self._corpus_by_intent = {}
        self._keywords_by_intent = {}
        self._load_all()

    def _load_json(self, path):
        full = self._base / path
        if not full.exists():
            raise FileNotFoundError(f"JSON no encontrado: {full}")
        with open(full, "r", encoding="utf-8") as f:
            return json.load(f)

    def _load_all(self):
        self._corpus = self._load_json("corpus/corpus_general.json")
        intents_raw = self._load_json("knowledge_base/intents.json")
        self._intents = intents_raw.get("intents", intents_raw) if isinstance(intents_raw, dict) else intents_raw
        kw_raw = self._load_json("knowledge_base/keywords.json")
        self._keywords = kw_raw.get("keywords", kw_raw) if isinstance(kw_raw, dict) else kw_raw
        self._faq_index = self._load_json("knowledge_base/faq_index.json")
        self._tutorias = self._load_json("knowledge_files/tutorias.json")
        self._practicas = self._load_json("knowledge_files/practicas.json")
        self._servicios = self._load_json("knowledge_files/servicios.json")
        for e in self._corpus:
            self._corpus_by_id[e["id"]] = e
            self._corpus_by_intent[e["intent"]] = e
        for kw in self._keywords:
            self._keywords_by_intent[kw["intent_id"]] = kw
        print(f"  [KB] {len(self._corpus)} entradas | "
              f"{len(self._intents)} intents | {len(self._keywords)} keywords")

    def get_by_intent(self, intent_id):
        rapid = self._faq_index.get("busqueda_rapida", {})
        entry_id = rapid.get(intent_id)
        if entry_id:
            return self._corpus_by_id.get(entry_id)
        return self._corpus_by_intent.get(intent_id)

    def get_by_id(self, entry_id):
        return self._corpus_by_id.get(entry_id)

    def get_by_category(self, cat_id):
        return [e for e in self._corpus if e["categoria"] == cat_id]

    def get_keywords(self, intent_id):
        return self._keywords_by_intent.get(intent_id)

    def get_all_keywords(self):
        return self._keywords

    def get_normative_detail(self, categoria):
        if categoria in ("TUT", "TTUT"): return self._tutorias
        elif categoria == "PPP": return self._practicas
        elif categoria in ("BIE", "MOV", "SER"): return self._servicios
        return None

    def get_confianza_minima(self, intent_id):
        for i in self._intents:
            if i["intent_id"] == intent_id:
                return i.get("confianza_minima", 0.72)
        return 0.72

    @property
    def categories(self):
        return sorted({e["categoria"] for e in self._corpus})



## IntentMatcher

In [15]:

# ═══════════════════════════════════════════════════
# CELDA 6 — IntentMatcher PRO (con stemming, fuzzy y contexto)
# ═══════════════════════════════════════════════════
import math
from difflib import SequenceMatcher


class IntentResult:
    def __init__(self, intent_id, score, categoria, suggestions=None):
        self.intent_id = intent_id
        self.score = score
        self.categoria = categoria
        self.matched = intent_id is not None
        self.suggestions = suggestions or []  # intents alternativos

    def __repr__(self):
        return (f"IntentResult(intent='{self.intent_id}', "
                f"score={self.score:.3f}, cat='{self.categoria}')")


class IntentMatcher:
    """
    Algoritmo de 4 niveles:
      Nivel 1: Trigger phrases — alta precisión (score 1.0)
      Nivel 2: Phrase match exacto (bigrama/trigrama)
      Nivel 3: Token match con IDF + stemming
      Nivel 4: Fuzzy match (n-gramas de caracteres) para typos

    Características Pro:
      • Stemming español ligero
      • Contexto conversacional (boost a la categoría anterior)
      • Sugerencias de intents alternativos cuando hay ambigüedad
      • Detección de typos con n-gramas
    """

    W_PRIMARIO = 3.0
    W_SECUNDARIO = 1.5
    W_SINONIMO = 2.0
    W_PHRASE_BONUS = 2.5

    THRESHOLD_MIN = 0.05
    THRESHOLD_HIGH = 0.40
    CONTEXT_BOOST = 1.20  # 20% boost a la última categoría

    def __init__(self, kb, preprocessor):
        self._kb = kb
        self._prep = preprocessor
        self._idf = {}
        self._stem_index = {}     # token raíz → tokens originales
        self._char_ngrams = {}    # token → ngrams
        self._last_category = None
        self._build_indices()

    def _build_indices(self):
        """Construye índices: IDF, stems y n-gramas para todas las keywords."""
        N = len(self._kb.get_all_keywords())
        df = {}
        all_tokens = set()

        for kw_entry in self._kb.get_all_keywords():
            toks_entry = set()
            for grupo in ("keywords_primarios","keywords_secundarios",
                          "sinonimos","frases_trigger"):
                for kw in kw_entry.get(grupo, []):
                    for t in self._prep.normalize(kw).split():
                        if len(t) > 2 and t not in self._prep.STOPWORDS:
                            toks_entry.add(t)
                            all_tokens.add(t)
            for t in toks_entry:
                df[t] = df.get(t, 0) + 1

        # IDF
        self._idf = {t: math.log((N + 1) / (cnt + 1)) + 1
                     for t, cnt in df.items()}

        # Stem index
        for t in all_tokens:
            stem = self._prep.stem(t)
            self._stem_index.setdefault(stem, set()).add(t)

        # Char n-grams (para fuzzy)
        for t in all_tokens:
            if len(t) >= 4:
                self._char_ngrams[t] = self._make_ngrams(t)

    def _make_ngrams(self, word, n=3):
        if len(word) < n:
            return {word}
        return {word[i:i+n] for i in range(len(word) - n + 1)}

    def _fuzzy_lookup(self, token, threshold=0.78):
        """
        Busca el token conocido más similar (fuzzy) usando SequenceMatcher.
        Detecta typos comunes: transposiciones, omisiones, sustituciones.
        """
        if token in self._idf:
            return token, 1.0
        if len(token) < 4:
            return None, 0
        best, best_sim = None, 0
        # Pre-filtro: solo candidatos de longitud similar
        for kw in self._char_ngrams.keys():
            if abs(len(kw) - len(token)) > 3:
                continue
            sim = SequenceMatcher(None, token, kw).ratio()
            if sim > best_sim:
                best, best_sim = kw, sim
        if best_sim >= threshold:
            return best, best_sim
        return None, 0

    def _idf_of(self, t):
        return self._idf.get(t, 1.0)

    def _score_keyword(self, keyword, user_tokens, user_expanded, user_norm):
        """Score de una keyword vs. consulta del usuario."""
        kw_norm = self._prep.normalize(keyword)
        kw_tokens = [t for t in kw_norm.split()
                     if t not in self._prep.STOPWORDS and len(t) > 2]
        if not kw_tokens:
            return 0.0

        # Bonus por frase exacta
        if len(kw_tokens) >= 2 and kw_norm in user_norm:
            avg_idf = sum(self._idf_of(t) for t in kw_tokens) / len(kw_tokens)
            return avg_idf * self.W_PHRASE_BONUS

        # Match directo o vía stem
        matched = []
        for kt in kw_tokens:
            if kt in user_tokens:
                matched.append(kt)
            elif kt in user_expanded:
                matched.append(kt)
            else:
                # Probar stem
                kt_stem = self._prep.stem(kt)
                if kt_stem in user_expanded:
                    matched.append(kt)

        if not matched:
            return 0.0

        overlap = len(matched) / len(kw_tokens)
        avg_idf = sum(self._idf_of(t) for t in matched) / len(matched)
        return overlap * avg_idf

    def _score_entry(self, kw_entry, normalized, tokens, expanded):
        # FASE 1: trigger phrases (con bonificación por especificidad)
        best_trigger_score = 0.0
        for trigger in kw_entry.get("frases_trigger", []):
            t_norm = self._prep.normalize(trigger)
            t_toks = set(t for t in t_norm.split()
                         if t not in self._prep.STOPWORDS and len(t) > 2)
            if not t_toks:
                continue
            matched_via_stem = 0
            for tt in t_toks:
                if tt in tokens or tt in expanded:
                    matched_via_stem += 1
                elif self._prep.stem(tt) in expanded:
                    matched_via_stem += 1
            if matched_via_stem == len(t_toks):
                # Coincidencia literal de frase → 1.0
                if t_norm in normalized:
                    return 1.0
                # Bonificación por especificidad: más tokens → mejor score
                # 1 token: 0.80 | 2 tokens: 0.92 | 3+ tokens: 0.96
                specificity_bonus = 0.75 + 0.07 * min(len(t_toks), 4)
                best_trigger_score = max(best_trigger_score, specificity_bonus)

        if best_trigger_score > 0:
            return best_trigger_score

        # FASE 2 + 3: scoring ponderado
        primarios = kw_entry.get("keywords_primarios", [])
        secundarios = kw_entry.get("keywords_secundarios", [])
        sinonimos = kw_entry.get("sinonimos", [])

        sp = sum(self._score_keyword(k, tokens, expanded, normalized)
                 for k in primarios)
        ss = sum(self._score_keyword(k, tokens, expanded, normalized)
                 for k in secundarios)
        sy = sum(self._score_keyword(k, tokens, expanded, normalized)
                 for k in sinonimos)

        weighted = (sp * self.W_PRIMARIO +
                    ss * self.W_SECUNDARIO +
                    sy * self.W_SINONIMO)

        max_possible = (len(primarios) * self.W_PRIMARIO +
                        len(secundarios) * self.W_SECUNDARIO +
                        len(sinonimos) * self.W_SINONIMO) * 2.5

        if max_possible == 0:
            return 0.0
        return min(weighted / max_possible, 0.85)

    def detect(self, user_text):
        normalized, tokens, expanded = self._prep.process(user_text)
        if not tokens:
            return IntentResult(None, 0.0, None)

        # Fuzzy correction: intentar reemplazar tokens desconocidos
        fixed_tokens = set(tokens)
        for t in tokens:
            if t not in self._idf:
                fix, sim = self._fuzzy_lookup(t)
                if fix:
                    fixed_tokens.add(fix)
        expanded = expanded | fixed_tokens

        # Calcular score para todos los intents
        scores = []
        for kw_entry in self._kb.get_all_keywords():
            score = self._score_entry(kw_entry, normalized, fixed_tokens, expanded)
            if score > 0:
                cat = kw_entry.get("categoria")
                # Boost por contexto conversacional
                if self._last_category and cat == self._last_category:
                    score *= self.CONTEXT_BOOST
                scores.append((score, kw_entry["intent_id"], cat))

        if not scores:
            return IntentResult(None, 0.0, None)

        scores.sort(key=lambda x: -x[0])
        best_score, best_intent, best_cat = scores[0]

        intent_min_conf = self._kb.get_confianza_minima(best_intent)

        # Sugerencias: top 3 alternativos
        suggestions = []
        for s, iid, c in scores[1:4]:
            if s >= 0.10:
                suggestions.append({"intent_id": iid, "score": s, "categoria": c})

        if best_score < self.THRESHOLD_MIN:
            return IntentResult(None, best_score, None, suggestions)

        # Verificar confianza mínima del intent específico
        if best_score < intent_min_conf:
            return IntentResult(None, best_score, None, suggestions)

        # Detectar ambigüedad
        if len(scores) > 1:
            second_score = scores[1][0]
            ratio = second_score / best_score if best_score > 0 else 0
            # Si están muy cerca y el mejor es bajo → ambiguo
            if ratio > 0.92 and best_score < 0.30:
                return IntentResult(None, best_score, None, suggestions)

        # Actualizar contexto
        self._last_category = best_cat
        return IntentResult(best_intent, best_score, best_cat, suggestions)

    def reset_context(self):
        self._last_category = None


## ResponseBuilder

In [16]:

# ═══════════════════════════════════════════════════
# CELDA 7 — ResponseBuilder mejorado
# ═══════════════════════════════════════════════════
class ResponseBuilder:
    CAT_LABELS = {
        "TUT":  "Proceso de Tutorías",
        "TTUT": "Tipos de Tutoría",
        "CUR":  "Cursos y Semestres",
        "ESP":  "Especialidades",
        "PPP":  "Prácticas Pre-Profesionales",
        "BIE":  "Bienestar Universitario",
        "MOV":  "Movilidad Estudiantil",
        "MAT":  "Matrícula",
        "TIT":  "Titulación",
        "SER":  "Servicios Universitarios"
    }

    CAT_COLORS = {
        "TUT":  "#1976D2", "TTUT": "#1976D2",
        "CUR":  "#7B1FA2", "ESP":  "#C2185B",
        "PPP":  "#F57C00", "BIE":  "#388E3C",
        "MOV":  "#0097A7", "MAT":  "#D32F2F",
        "TIT":  "#5D4037", "SER":  "#455A64"
    }

    CAT_EMOJI = {
        "TUT":  "📚", "TTUT": "📌", "CUR":  "📖", "ESP":  "🎯",
        "PPP":  "💼", "BIE":  "🏥", "MOV":  "✈️", "MAT":  "📋",
        "TIT":  "🎓", "SER":  "🖥️"
    }

    def __init__(self, kb):
        self._kb = kb

    def build(self, intent_result):
        """Construye respuesta estructurada."""
        if not intent_result.matched:
            return {
                "tipo": "fallback",
                "texto": self._fallback_text(intent_result),
                "categoria": None,
                "pregunta": None,
                "fuente": None,
                "sugerencias": intent_result.suggestions or []
            }

        entry = self._kb.get_by_intent(intent_result.intent_id)
        if not entry:
            return {"tipo": "fallback",
                    "texto": self._fallback_text(intent_result),
                    "categoria": None, "pregunta": None,
                    "fuente": None, "sugerencias": []}

        respuesta = entry.get("respuesta", "")
        enriq = self._enriquecimiento(entry)

        return {
            "tipo": "respuesta",
            "texto": respuesta,
            "enriquecimiento": enriq,
            "categoria": entry.get("categoria"),
            "categoria_nombre": entry.get("categoria_nombre",
                                          self.CAT_LABELS.get(entry.get("categoria"), "")),
            "pregunta": entry.get("pregunta", ""),
            "fuente": entry.get("fuente", "—"),
            "id_corpus": entry.get("id", ""),
            "score": intent_result.score,
            "sugerencias": intent_result.suggestions or []
        }

    def _enriquecimiento(self, entry):
        cat = entry.get("categoria", "")
        intent = entry.get("intent", "")
        norm = self._kb.get_normative_detail(cat)
        if not norm:
            return ""
        if cat in ("TUT","TTUT"):
            if "confidencial" in intent:
                c = norm.get("confidencialidad", {})
                return f"{c.get('art','')} — {c.get('deber','')}"
            if "frecuencia" in intent:
                p = norm.get("periodicidad", {})
                m = p.get("momentos_clave_semestre", [])
                if m: return "Momentos clave: " + " | ".join(m)
            if "cambio_tutor" in intent:
                a = norm.get("asignacion_tutores", {})
                return (f"{a.get('art','')} — Máx tutorados: "
                        f"{a.get('maximo_tutorados', 25)}")
        elif cat == "PPP":
            if "remuneracion" in intent:
                r = norm.get("remuneracion", {})
                return f"{r.get('norma','')} — {r.get('subvencion_minima','')}"
            if "requisito" in intent and "titulacion" in intent:
                r = norm.get("relacion_titulacion", {})
                return r.get("descripcion","")
        return ""

    def _fallback_text(self, intent_result):
        return ("No estoy seguro de tu consulta. ¿Podrías reformularla con "
                "más detalle? Puedes preguntarme sobre tutorías, matrícula, "
                "titulación, prácticas, especialidades o servicios UNSAAC.")

    def get_pregunta_for_intent(self, intent_id):
        entry = self._kb.get_by_intent(intent_id)
        return entry.get("pregunta", "") if entry else ""


## ConversationHistory

In [17]:

# ═══════════════════════════════════════════════════
# CELDA 8 — ConversationHistory
# ═══════════════════════════════════════════════════
from datetime import datetime


class ConversationHistory:
    def __init__(self):
        self._turnos = []
        self._session_start = datetime.now()

    def record(self, user_input, normalized, intent_result, respuesta_dict):
        self._turnos.append({
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "user_input": user_input,
            "normalized": normalized,
            "intent_id": intent_result.intent_id,
            "score": round(intent_result.score, 4),
            "categoria": intent_result.categoria,
            "fue_fallback": not intent_result.matched,
            "respuesta": respuesta_dict
        })

    @property
    def total_turnos(self): return len(self._turnos)

    @property
    def tasa_resolucion(self):
        if not self._turnos: return 0.0
        ok = sum(1 for t in self._turnos if not t["fue_fallback"])
        return round(ok / len(self._turnos) * 100, 1)

    @property
    def tasa_fallback(self):
        return round(100 - self.tasa_resolucion, 1)

    def distribucion_categorias(self):
        d = {}
        for t in self._turnos:
            if t["categoria"]:
                d[t["categoria"]] = d.get(t["categoria"], 0) + 1
        return dict(sorted(d.items(), key=lambda x: -x[1]))

    def export_report(self):
        d = self.distribucion_categorias()
        lines = [
            "=" * 60,
            "  REPORTE DE EVALUACIÓN — CHATBOT EPIIS-UNSAAC PRO",
            "=" * 60,
            f"  Sesión iniciada : {self._session_start.strftime('%Y-%m-%d %H:%M:%S')}",
            f"  Total turnos    : {self.total_turnos}",
            f"  Tasa resolución : {self.tasa_resolucion}%",
            f"  Tasa fallback   : {self.tasa_fallback}%",
            "",
            "  Distribución por categoría:",
        ]
        for cat, cnt in d.items():
            pct = round(cnt / max(self.total_turnos, 1) * 100, 1)
            bar = "█" * int(pct / 5)
            lines.append(f"    {cat:6s} {bar:<20s} {cnt:3d} ({pct}%)")
        return "\n".join(lines)


## ChatbotEPIIS - Frontend

In [18]:

# ═══════════════════════════════════════════════════
# CELDA 9 — ChatbotEPIIS (fachada)
# ═══════════════════════════════════════════════════
class ChatbotEPIIS:
    def __init__(self, base_dir="."):
        print("Inicializando Chatbot EPIIS-UNSAAC PRO v2.0...")
        self._preprocessor = Preprocessor()
        self._kb = KnowledgeBaseLoader(base_dir)
        self._matcher = IntentMatcher(self._kb, self._preprocessor)
        self._builder = ResponseBuilder(self._kb)
        self._history = ConversationHistory()
        print(f"  [Algoritmo] IDF: {len(self._matcher._idf)} términos | "
              f"Stems: {len(self._matcher._stem_index)} | "
              f"Ngrams: {len(self._matcher._char_ngrams)}")
        print("✓ Sistema listo.\n")

    def respond(self, user_input):
        if not user_input.strip():
            return {"tipo": "fallback", "texto": "Por favor, escribe tu consulta.",
                    "categoria": None, "pregunta": None, "fuente": None,
                    "sugerencias": []}
        normalized, _, _ = self._preprocessor.process(user_input)
        result = self._matcher.detect(user_input)
        respuesta = self._builder.build(result)
        self._history.record(user_input, normalized, result, respuesta)
        return respuesta

    def reset_context(self):
        self._matcher.reset_context()

    def get_pregunta(self, intent_id):
        return self._builder.get_pregunta_for_intent(intent_id)

    @property
    def kb(self): return self._kb

    @property
    def builder(self): return self._builder

    @property
    def history(self): return self._history

    @property
    def stats(self):
        return {
            "turnos": self._history.total_turnos,
            "tasa_resolucion": self._history.tasa_resolucion,
            "tasa_fallback": self._history.tasa_fallback,
            "distribucion": self._history.distribucion_categorias()
        }


## Inicialización


In [19]:

# ═══════════════════════════════════════════════════
# CELDA 10 — Inicialización y verificación
# ═══════════════════════════════════════════════════
bot = ChatbotEPIIS()
#bot.start_session()

print(f"Categorías disponibles : {bot._kb.categories}")
print(f"Entradas corpus        : {len(bot._kb._corpus)}")
print(f"Keywords cargadas      : {len(bot._kb._keywords)}")
print(f"Términos en índice IDF : {len(bot._matcher._idf)}")


Inicializando Chatbot EPIIS-UNSAAC PRO v2.0...
  [KB] 100 entradas | 100 intents | 100 keywords
  [Algoritmo] IDF: 647 términos | Stems: 618 | Ngrams: 631
✓ Sistema listo.

Categorías disponibles : ['BIE', 'CUR', 'ESP', 'MAT', 'MOV', 'PPP', 'SER', 'TIT', 'TTUT', 'TUT']
Entradas corpus        : 100
Keywords cargadas      : 100
Términos en índice IDF : 647


In [20]:

# ═══════════════════════════════════════════════════
# CELDA 10 — Inicialización
# ═══════════════════════════════════════════════════

bot = ChatbotEPIIS()
print(f"Categorías : {bot.kb.categories}")

Inicializando Chatbot EPIIS-UNSAAC PRO v2.0...
  [KB] 100 entradas | 100 intents | 100 keywords
  [Algoritmo] IDF: 647 términos | Stems: 618 | Ngrams: 631
✓ Sistema listo.

Categorías : ['BIE', 'CUR', 'ESP', 'MAT', 'MOV', 'PPP', 'SER', 'TIT', 'TTUT', 'TUT']


## Casos de prueba automáticos

In [21]:
# ═══════════════════════════════════════════════════
# CELDA 11 — Casos de prueba automáticos
# ═══════════════════════════════════════════════════
CASOS_PRUEBA = [
    ("¿Qué es la tutoría académica en la UNSAAC?", "consulta_tutoria_definicion"),
    ("¿Con qué frecuencia debo asistir a tutoría?", "consulta_tutoria_frecuencia"),
    ("¿Las sesiones de tutoría son confidenciales?", "consulta_tutoria_confidencialidad"),
    ("¿Puedo cambiar de tutor?", "consulta_tutoria_cambio_tutor"),
    ("cuales son los tipos de tutoria", "consulta_tipos_tutoria_general"),
    ("hay tutoria virtual o en linea", "consulta_tipos_tutoria_virtual"),
    ("tutoría para estudiantes con bajo rendimiento", "consulta_tipos_tutoria_riesgo_academico"),
    ("qué diferencia hay entre tutor y asesor de tesis", "consulta_tipos_tutoria_vs_asesoria_tesis"),
    ("cuantos semestres dura la carrera de sistemas", "consulta_cursos_duracion_carrera"),
    ("cuantos creditos necesito para graduarme", "consulta_cursos_total_creditos"),
    ("qué pasa si jalo un curso", "consulta_cursos_desaprobacion"),
    ("cómo se calcula el promedio ponderado", "consulta_cursos_promedio_ponderado"),
    ("que especialidades tiene la epiis", "consulta_esp_listado_especialidades"),
    ("cuando tengo que elegir mi especialidad", "consulta_esp_cuando_elegir"),
    ("que se estudia en inteligencia artificial y datos", "consulta_esp_ia_ciencia_datos"),
    ("cuales son los requisitos para las practicas ppp", "consulta_ppp_requisitos_inicio"),
    ("cuantas horas de practicas debo hacer", "consulta_ppp_horas_acumuladas"),
    ("me pagan en las practicas pre profesionales", "consulta_ppp_remuneracion"),
    ("practicas son requisito para titularme", "consulta_ppp_requisito_titulacion"),
    ("que servicios de bienestar ofrece la unsaac", "consulta_bie_servicios_generales"),
    ("como accedo al psicologo gratuito", "consulta_bie_atencion_psicologica"),
    ("como solicito la beca de alimentacion", "consulta_bie_beca_alimentacion"),
    ("que programas de intercambio hay", "consulta_mov_programas_disponibles"),
    ("requisitos para el intercambio estudiantil", "consulta_mov_requisitos_intercambio"),
    ("me reconocen los cursos del intercambio", "consulta_mov_convalidacion_cursos"),
    ("cuando es la matricula en la unsaac", "consulta_mat_fechas_matricula"),
    ("como me matriculo en serunsa", "consulta_mat_proceso_serunsa"),
    ("que es la matricula condicional", "consulta_mat_condicional"),
    ("cuantos creditos maximo puedo llevar", "consulta_mat_creditos_maximo"),
    ("cuales son las modalidades de titulacion", "consulta_tit_modalidades"),
    ("como obtengo el grado de bachiller", "consulta_tit_grado_bachiller"),
    ("cuanto tarda el proceso de titulacion", "consulta_tit_duracion_proceso"),
    ("donde registro mi titulo en sunedu", "consulta_tit_registro_sunedu"),
    ("como accedo al aula virtual moodle", "consulta_ser_aula_virtual"),
    ("como saco mi correo institucional", "consulta_ser_correo_institucional"),
    ("que es el tupa", "consulta_ser_tupa"),
    # Variaciones léxicas y typos
    ("necesito saber sobre las ppp", "consulta_ppp_requisitos_inicio"),
    ("jale un curso qué hago", "consulta_cursos_desaprobacion"),
    ("quiero cambiar de tutor", "consulta_tutoria_cambio_tutor"),
    ("matricla en serunsa", "consulta_mat_proceso_serunsa"),  # typo
    ("tutoria virtaul", "consulta_tipos_tutoria_virtual"),    # typo
]

bot.reset_context()
ok = 0
print(f"\n{'#':>3} {'STATUS':8} {'SCORE':6} {'ESPERADO':42} {'OBTENIDO':42}")
print("-" * 110)
for i, (q, esp) in enumerate(CASOS_PRUEBA, 1):
    r = bot._matcher.detect(q)
    status = "✓ OK   " if r.intent_id == esp else "✗ FAIL "
    if r.intent_id == esp: ok += 1
    print(f"{i:>3} {status} {r.score:6.3f} {str(esp):42s} {str(r.intent_id):42s}")
bot.reset_context()
total = len(CASOS_PRUEBA)
print("=" * 110)
print(f"\n  Aciertos: {ok}/{total} ({ok/total*100:.1f}%)")




  # STATUS   SCORE  ESPERADO                                   OBTENIDO                                  
--------------------------------------------------------------------------------------------------------------
  1 ✓ OK     1.000 consulta_tutoria_definicion                consulta_tutoria_definicion               
  2 ✗ FAIL   0.984 consulta_tutoria_frecuencia                consulta_tutoria_definicion               
  3 ✓ OK     1.068 consulta_tutoria_confidencialidad          consulta_tutoria_confidencialidad         
  4 ✓ OK     1.068 consulta_tutoria_cambio_tutor              consulta_tutoria_cambio_tutor             
  5 ✗ FAIL   0.984 consulta_tipos_tutoria_general             consulta_tutoria_definicion               
  6 ✓ OK     1.000 consulta_tipos_tutoria_virtual             consulta_tipos_tutoria_virtual            
  7 ✓ OK     1.152 consulta_tipos_tutoria_riesgo_academico    consulta_tipos_tutoria_riesgo_academico   
  8 ✓ OK     1.152 consulta_tipos_tutoria_vs_as

## 💬 Chat Interactivo

Ejecuta la celda siguiente para iniciar el chat.  
Escribe cualquier consulta académica. Comandos especiales:
- `ayuda` → ver categorías disponibles
- `historial` → ver turnos de la sesión  
- `reporte` → exportar reporte de evaluación
- `salir` → terminar el chat


In [ ]:

# ═══════════════════════════════════════════════════
# CELDA 12 — Interfaz Pro (HTML + ipywidgets)
# ═══════════════════════════════════════════════════
from IPython.display import HTML, display, clear_output
import ipywidgets as widgets
import html as htmlmod


# CSS Profesional
CHAT_CSS = """
<style>
.epiis-chat {
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    max-width: 760px;
    margin: 0 auto;
}
.epiis-header {
    background: linear-gradient(135deg, #1E568A 0%, #2E86C1 100%);
    color: white;
    padding: 18px 22px;
    border-radius: 14px 14px 0 0;
    display: flex;
    align-items: center;
    gap: 14px;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
}
.epiis-header-icon {
    font-size: 32px;
    background: rgba(255,255,255,0.18);
    width: 52px; height: 52px;
    border-radius: 50%;
    display: flex; align-items: center; justify-content: center;
}
.epiis-header-text h3 { margin: 0; font-size: 17px; font-weight: 600; }
.epiis-header-text p { margin: 2px 0 0; font-size: 12px; opacity: 0.92; }
.epiis-status {
    margin-left: auto;
    background: #4CAF50;
    color: white;
    padding: 4px 11px;
    border-radius: 12px;
    font-size: 11px;
    display: flex; align-items: center; gap: 5px;
}
.epiis-status::before {
    content: ''; width: 6px; height: 6px;
    border-radius: 50%; background: white;
    animation: pulse 1.5s infinite;
}
@keyframes pulse { 0%,100%{opacity:1;} 50%{opacity:.4;} }

.epiis-msgs {
    background: #F5F7FA;
    padding: 18px;
    min-height: 320px;
    max-height: 560px;
    overflow-y: auto;
    border-left: 1px solid #E1E5EA;
    border-right: 1px solid #E1E5EA;
}
.epiis-msg {
    display: flex;
    margin-bottom: 16px;
    align-items: flex-start;
    animation: slideIn 0.3s ease-out;
}
@keyframes slideIn {
    from { opacity: 0; transform: translateY(8px); }
    to { opacity: 1; transform: translateY(0); }
}
.epiis-msg.user { flex-direction: row-reverse; }
.epiis-avatar {
    width: 38px; height: 38px;
    border-radius: 50%;
    display: flex; align-items: center; justify-content: center;
    font-size: 19px;
    flex-shrink: 0;
    background: #FFF;
    box-shadow: 0 1px 3px rgba(0,0,0,0.12);
}
.epiis-msg.user .epiis-avatar { background: #1E568A; color: white; }
.epiis-msg.bot .epiis-avatar { background: white; }
.epiis-bubble {
    margin: 0 11px;
    padding: 12px 16px;
    border-radius: 16px;
    max-width: 78%;
    word-wrap: break-word;
    box-shadow: 0 1px 2px rgba(0,0,0,0.08);
    line-height: 1.5;
    font-size: 14.5px;
}
.epiis-msg.user .epiis-bubble {
    background: #1E568A;
    color: white;
    border-bottom-right-radius: 4px;
}
.epiis-msg.bot .epiis-bubble {
    background: white;
    color: #202124;
    border-bottom-left-radius: 4px;
}
.epiis-bubble.fallback {
    background: #FFF8E1;
    border-left: 4px solid #FFB300;
}
.epiis-badge {
    display: inline-block;
    padding: 3px 11px;
    border-radius: 14px;
    font-size: 11px;
    font-weight: 600;
    color: white;
    margin-bottom: 9px;
    letter-spacing: 0.3px;
}
.epiis-question {
    font-size: 12.5px;
    color: #5F6368;
    font-style: italic;
    margin-bottom: 8px;
    padding-bottom: 8px;
    border-bottom: 1px dashed #E1E5EA;
}
.epiis-answer { white-space: pre-wrap; }
.epiis-enriq {
    background: #E8F0FE;
    border-left: 3px solid #1E88E5;
    padding: 8px 11px;
    margin-top: 10px;
    border-radius: 4px;
    font-size: 13px;
    color: #1A56A3;
}
.epiis-source {
    margin-top: 10px;
    padding-top: 9px;
    border-top: 1px solid #EFF1F4;
    font-size: 11.5px;
    color: #5F6368;
    display: flex;
    align-items: center;
    gap: 5px;
}
.epiis-confidence {
    margin-top: 10px;
    display: flex;
    align-items: center;
    gap: 8px;
    font-size: 11px;
    color: #5F6368;
}
.epiis-conf-bar {
    flex: 1;
    height: 5px;
    background: #E0E0E0;
    border-radius: 3px;
    overflow: hidden;
}
.epiis-conf-fill {
    height: 100%;
    background: linear-gradient(90deg, #FF7043, #FFCA28, #66BB6A);
    border-radius: 3px;
    transition: width 0.5s;
}
.epiis-suggestions {
    margin-top: 12px;
    padding-top: 10px;
    border-top: 1px solid #EFF1F4;
}
.epiis-suggestions-label {
    font-size: 11px;
    color: #5F6368;
    margin-bottom: 7px;
    text-transform: uppercase;
    letter-spacing: 0.5px;
    font-weight: 600;
}
.epiis-sugg-chip {
    display: inline-block;
    background: #F1F3F4;
    border: 1px solid #DADCE0;
    padding: 6px 12px;
    border-radius: 14px;
    font-size: 12px;
    color: #1A73E8;
    margin: 3px 4px 3px 0;
    cursor: pointer;
    transition: all 0.2s;
}
.epiis-sugg-chip:hover {
    background: #E8F0FE;
    border-color: #1A73E8;
}
.epiis-footer {
    background: white;
    border: 1px solid #E1E5EA;
    border-top: none;
    border-radius: 0 0 14px 14px;
    padding: 14px;
}
.epiis-quick {
    display: flex;
    flex-wrap: wrap;
    gap: 7px;
    margin-bottom: 10px;
}
.epiis-quick-btn {
    background: #F1F3F4;
    border: 1px solid #DADCE0;
    color: #1A73E8;
    padding: 6px 13px;
    border-radius: 16px;
    font-size: 12px;
    cursor: pointer;
}
.epiis-empty {
    text-align: center;
    color: #80868B;
    padding: 40px 20px;
}
.epiis-empty-icon { font-size: 48px; margin-bottom: 10px; }
.epiis-empty p { margin: 4px 0; font-size: 13px; }
.epiis-typing {
    display: flex;
    gap: 4px;
    padding: 12px 16px;
    background: white;
    border-radius: 16px;
    width: fit-content;
    margin-left: 49px;
}
.epiis-typing span {
    width: 8px; height: 8px;
    background: #80868B;
    border-radius: 50%;
    animation: bounce 1.4s infinite;
}
.epiis-typing span:nth-child(2) { animation-delay: 0.2s; }
.epiis-typing span:nth-child(3) { animation-delay: 0.4s; }
@keyframes bounce {
    0%,60%,100% { transform: translateY(0); }
    30% { transform: translateY(-8px); }
}
</style>
"""


CAT_COLORS_UI = {
    "TUT":"#1976D2","TTUT":"#1976D2","CUR":"#7B1FA2","ESP":"#C2185B",
    "PPP":"#F57C00","BIE":"#388E3C","MOV":"#0097A7","MAT":"#D32F2F",
    "TIT":"#5D4037","SER":"#455A64"
}
CAT_EMOJI_UI = {
    "TUT":"📚","TTUT":"📌","CUR":"📖","ESP":"🎯","PPP":"💼",
    "BIE":"🏥","MOV":"✈️","MAT":"📋","TIT":"🎓","SER":"🖥️"
}

PREGUNTAS_RAPIDAS = [
    "¿Qué es la tutoría académica?",
    "¿Cómo me matriculo en SERUNSA?",
    "¿Cuántos créditos para graduarme?",
    "¿Modalidades de titulación?",
    "¿Requisitos de prácticas pre-profesionales?",
    "¿Servicios de bienestar?",
]


def render_header():
    return """
    <div class="epiis-header">
      <div class="epiis-header-icon">🤖</div>
      <div class="epiis-header-text">
        <h3>Asistente Académico EPIIS-UNSAAC</h3>
        <p>Consultas sobre tutorías · matrícula · titulación · prácticas · más</p>
      </div>
      <div class="epiis-status">en línea</div>
    </div>
    """


def render_empty():
    return """
    <div class="epiis-empty">
      <div class="epiis-empty-icon">💬</div>
      <p><strong>¡Hola! Soy el Asistente Académico EPIIS</strong></p>
      <p>Hazme una pregunta o usa los botones rápidos para empezar.</p>
    </div>
    """


def render_user_msg(text):
    safe = htmlmod.escape(text)
    return f"""
    <div class="epiis-msg user">
      <div class="epiis-avatar">👤</div>
      <div class="epiis-bubble">{safe}</div>
    </div>
    """


def render_bot_msg(resp):
    cat = resp.get("categoria")
    color = CAT_COLORS_UI.get(cat, "#5F6368")
    emoji = CAT_EMOJI_UI.get(cat, "🤖")
    is_fallback = resp.get("tipo") == "fallback"
    bubble_cls = "epiis-bubble fallback" if is_fallback else "epiis-bubble"

    parts = []
    if is_fallback:
        parts.append(f'<div class="epiis-answer">{htmlmod.escape(resp["texto"])}</div>')
    else:
        if cat:
            cat_name = ResponseBuilder.CAT_LABELS.get(cat, cat)
            parts.append(
                f'<div class="epiis-badge" style="background:{color};">'
                f'{emoji} {cat_name}</div>'
            )
        if resp.get("pregunta"):
            parts.append(f'<div class="epiis-question">❓ {htmlmod.escape(resp["pregunta"])}</div>')
        parts.append(f'<div class="epiis-answer">{htmlmod.escape(resp["texto"])}</div>')
        if resp.get("enriquecimiento"):
            parts.append(f'<div class="epiis-enriq">ℹ️ {htmlmod.escape(resp["enriquecimiento"])}</div>')
        if resp.get("fuente"):
            parts.append(
                f'<div class="epiis-source">📌 <strong>Fuente:</strong> '
                f'{htmlmod.escape(resp["fuente"])}</div>'
            )
        if resp.get("score") is not None:
            # Validamos que el score no sea mayor a 100%
            pct = min(int(resp["score"] * 100), 100)
            parts.append(
                f'<div class="epiis-confidence">'
                f'<span>Confianza</span>'
                f'<div class="epiis-conf-bar">'
                f'<div class="epiis-conf-fill" style="width:{pct}%;"></div>'
                f'</div><span><strong>{pct}%</strong></span></div>'
            )

    sugg = resp.get("sugerencias", [])
    if sugg:
        chips = []
        for s in sugg[:3]:
            p = bot.get_pregunta(s["intent_id"])
            if p:
                chips.append(
                    f'<span class="epiis-sugg-chip">{htmlmod.escape(p)}</span>'
                )
        if chips:
            parts.append(
                f'<div class="epiis-suggestions">'
                f'<div class="epiis-suggestions-label">¿Te referías a...?</div>'
                f'{"".join(chips)}'
                f'</div>'
            )

    return f"""
    <div class="epiis-msg bot">
      <div class="epiis-avatar">{emoji}</div>
      <div class="{bubble_cls}">{"".join(parts)}</div>
    </div>
    """


class ChatUI:
    def __init__(self, bot):
        self.bot = bot
        self.messages = []  # [(role, content), ...]
        self.chat_out = widgets.Output()
        self.input_box = widgets.Text(
            placeholder="Escribe tu pregunta y presiona Enter...",
            layout=widgets.Layout(width="100%"),
        )
        self.send_btn = widgets.Button(
            description="Enviar", button_style="primary",
            icon="paper-plane", layout=widgets.Layout(width="100px")
        )
        self.clear_btn = widgets.Button(
            description="Limpiar", icon="trash",
            layout=widgets.Layout(width="100px")
        )

        # Quick replies
        self.quick_buttons = []
        for q in PREGUNTAS_RAPIDAS:
            btn = widgets.Button(
                description=q,
                layout=widgets.Layout(width="auto", margin="2px"),
                style={"button_color": "#F1F3F4"}
            )
            btn.on_click(self._make_quick_handler(q))
            self.quick_buttons.append(btn)

        self.send_btn.on_click(self._on_send)
        self.input_box.on_submit(lambda _: self._on_send(None))
        self.clear_btn.on_click(self._on_clear)

        self._render()

    def _make_quick_handler(self, question):
        def handler(_):
            self._process(question)
        return handler

    def _on_send(self, _):
        text = self.input_box.value.strip()
        if not text:
            return
        self.input_box.value = ""
        self._process(text)

    def _on_clear(self, _):
        self.messages = []
        self.bot.reset_context()
        self._render()

    def _process(self, user_text):
        self.messages.append(("user", user_text))
        resp = self.bot.respond(user_text)
        self.messages.append(("bot", resp))
        self._render()

    def _render(self):
        with self.chat_out:
            clear_output(wait=True)
            html_parts = [CHAT_CSS, '<div class="epiis-chat">']
            html_parts.append(render_header())
            html_parts.append('<div class="epiis-msgs" id="epiis-msgs">')
            if not self.messages:
                html_parts.append(render_empty())
            else:
                for role, content in self.messages:
                    if role == "user":
                        html_parts.append(render_user_msg(content))
                    else:
                        html_parts.append(render_bot_msg(content))
            html_parts.append('</div></div>')
            display(HTML("".join(html_parts)))

    def show(self):
        quick_box = widgets.HBox(self.quick_buttons,
                                  layout=widgets.Layout(flex_flow="row wrap",
                                                         margin="8px 0"))
        input_row = widgets.HBox([self.input_box, self.send_btn, self.clear_btn],
                                  layout=widgets.Layout(width="100%"))
        display(widgets.VBox([
            self.chat_out,
            widgets.HTML("<div style='font-size:11px;color:#5F6368;"
                         "margin:8px 0 4px;font-weight:600;"
                         "text-transform:uppercase;letter-spacing:.5px;'>"
                         "Preguntas frecuentes</div>"),
            quick_box,
            input_row
        ]))


# Instanciar y mostrar la interfaz Pro
ui = ChatUI(bot)
ui.show()

ModuleNotFoundError: No module named 'ipywidgets'